# JORT v7.1 Full Pipeline - Colab Notebook
=========================================
Complete pipeline: MD/JSON → Extraction → LLM Enhancement → Chunking → Rich JSON

**Features:**
- Dual input: Raw MD files OR pre-extracted JSON
- AraBERT classifier (`aubmindlab/bert-base-arabertv2`)
- OpenRouter LLM (`meta-llama/llama-3.3-70b-instruct`)
- Rich output: `legal_features`, `relations`, `code` metadata
- Downloadable results (`articles.json` + `chunks.json`)

**Author:** Based on JORT Parser v7.1
**Date:** 2026-04-29

In [ ]:
# Install dependencies
!pip install -q pyarabic camel-tools transformers torch scikit-learn requests json-repair

# Standard libs
import os
import json
import re
from pathlib import Path
from typing import Dict, List, Optional
import time

# ML/NLP libs
try:
    import torch
    from transformers import AutoTokenizer, AutoModel, pipeline
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    
try:
    from camel_tools.tagger import DefaultTagger
    from camel_tools.tokenizers.word import simple_word_tokenize
    from camel_tools.ner import NERecognizer
    CAMEL_TOOLS_AVAILABLE = True
except ImportError:
    CAMEL_TOOLS_AVAILABLE = False

try:
    import pyarabic.araby as araby
    PYARABIC_AVAILABLE = True
except ImportError:
    PYARABIC_AVAILABLE = False

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

print(f"Transformers: {TRANSFORMERS_AVAILABLE}")
print(f"Camel Tools: {CAMEL_TOOLS_AVAILABLE}")
print(f"PyArabic: {PYARABIC_AVAILABLE}")
print(f"Scikit-learn: {SKLEARN_AVAILABLE}")

In [ ]:
# =============================================================================
# CONFIGURATION - User must fill these
# =============================================================================

# OpenRouter API Key (get from https://openrouter.ai/keys)
# Options: Use esprit account or your own
OPENROUTER_API_KEY = "sk-or-v1-..."  # ← FILL THIS

# Model configurations
ARABERT_MODEL = "aubmindlab/bert-base-arabertv2"
OPENROUTER_MODEL = "meta-llama/llama-3.3-70b-instruct"

# Thresholds
ARABERT_CONFIDENCE_THRESHOLD = 0.70
NER_CONFIDENCE_THRESHOLD = 0.70

# Processing
BATCH_SIZE = 10  # Articles per batch for LLM calls
DELAY_BETWEEN_REQUESTS = 5  # Seconds between LLM calls

print("Configuration loaded!")
print(f"OpenRouter Key set: {'Yes' if OPENROUTER_API_KEY != 'sk-or-v1-...' else 'NO - FILL API KEY'}")

In [ ]:
# =============================================================================
# UTILITIES (from utils.py)
# =============================================================================

def normalize_arabic_text(text: str) -> str:
    """Normalize Arabic text: remove diacritics, standardize chars."""
    if not PYARABIC_AVAILABLE:
        text = re.sub(r'[\u064B-\u065F]', '', text)
        text = text.replace('\u0640', '')
        return text.strip()
    
    text = araby.strip_tashkeel(text)
    text = araby.strip_tatweel(text)
    text = araby.normalize_hamza(text)
    return text.strip()

def convert_arabic_number(text: str) -> str:
    """Convert Arabic/Hindi numerals to Western numerals."""
    ordinal_map = {
        "الأول": "1", "الثاني": "2", "الثالث": "3", "الرابع": "4", "الخامس": "5",
        "السادس": "6", "السابع": "7", "الثامن": "8", "التاسع": "9", "العاشر": "10",
        "الحادي عشر": "11", "الثاني عشر": "12", "الثالث عشر": "13",
        "الرابع عشر": "14", "الخامس عشر": "15", "السادس عشر": "16",
        "السابع عشر": "17", "الثامن عشر": "18", "التاسع عشر": "19",
        "العشرون": "20", "الوحيد": "1"
    }
    
    for ar, num in ordinal_map.items():
        if ar in text:
            return num
    
    m = re.search(r'الفصل\s+(\d+)', text)
    if m:
        return m.group(1)
    
    return "?"

def repair_json(llm_output: str) -> dict:
    """Parse potentially malformed JSON from LLM output."""
    try:
        import json_repair
        return json_repair.repair_json(llm_output)
    except:
        json_match = re.search(r'\{.*\}', llm_output, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(0))
            except:
                pass
    return {}

print("Utility functions loaded!")

In [ ]:
# =============================================================================
# ARTICLE EXTRACTOR (from extractor.py)
# =============================================================================

class ArticleExtractor:
    """Extract articles from Arabic legal MD files."""
    
    ARTICLE_PATTERNS = [
        r"\*\*الفصل\s+\d+\s*-\*\*",
        r"\*\*الفصل\s+الأول\s*-\*\*",
        r"^##\s*الفصل\s+\d+",
    ]
    
    def __init__(self, code_file: str):
        self.file_path = Path(code_file)
        self.content = self.file_path.read_text(encoding="utf-8")
        self.code_name = self.file_path.stem
        
        # Initialize Camel tagger if available
        self.tagger = None
        if CAMEL_TOOLS_AVAILABLE:
            try:
                self.tagger = DefaultTagger()
            except:
                pass
    
    def _normalize_article_num(self, text: str) -> str:
        return convert_arabic_number(text)
    
    def _extract_content(self, start: int, end: int) -> str:
        """Extract and clean article content."""
        text = self.content[start:end].strip()
        text = re.sub(r"\*\*", "", text)
        text = re.sub(r"<[^>]+>", "", text)
        
        first_dash = text.find("-")
        if first_dash > 0 and first_dash < 30:
            text = text[first_dash + 1:].strip()
        
        # Stop at structural headings
        lines = text.split("\n")
        content_lines = []
        for line in lines:
            stripped = line.strip()
            if re.match(r"^#+\s*(ال)?(باب|قسم|كتاب)\s+", stripped):
                break
            if re.match(r"^##+\s+", stripped) and not re.match(r"الفصل\s+", stripped):
                break
            content_lines.append(line)
        
        text = "\n".join(content_lines)
        text = re.sub(r"\s+", " ", text).strip()
        return text[:2000]
    
    def _find_hierarchy(self, text_before: str) -> dict:
        """Find hierarchy (book/title/section) from text before article."""
        lines = text_before.split("\n")
        hierarchy = {"book": None, "title": None, "section": None}
        
        found_book = False
        found_title = False
        found_section = False
        
        for line in reversed(lines):
            if found_book and found_title and found_section:
                break
            
            l = line.strip()
            
            if not found_section and l.startswith("#### "):
                hierarchy["section"] = re.sub(r"\*+", "", l[5:]).strip()
                found_section = True
            elif not found_title and l.startswith("### "):
                hierarchy["title"] = re.sub(r"\*+", "", l[4:]).strip()
                found_title = True
            elif not found_book and l.startswith("## "):
                hierarchy["book"] = re.sub(r"\*+", "", l[3:]).strip()
                found_book = True
        
        return hierarchy
    
    def extract_all(self) -> dict:
        """Extract all articles with regex boundaries."""
        articles = []
        positions = []
        seen_positions = set()
        
        for pattern in self.ARTICLE_PATTERNS:
            for m in re.finditer(pattern, self.content, re.MULTILINE):
                pos = m.start()
                if any(abs(pos - p) < 5 for p in [p[0] for p in positions]):
                    continue
                if pos not in [p[0] for p in positions]:
                    positions.append((pos, m.group(0)))
                    seen_positions.add(pos)
        
        positions.sort(key=lambda x: x[0])
        
        for idx, (start, pat) in enumerate(positions):
            article_num = self._normalize_article_num(self.content[start:start + 30])
            end = positions[idx + 1][0] if idx + 1 < len(positions) else len(self.content)
            content = self._extract_content(start, end)
            hier = self._find_hierarchy(self.content[:start])
            
            articles.append({
                "article_number": article_num,
                "content": content,
                "page_num": 1,
                "book": hier.get("book"),
                "title": hier.get("title"),
                "section": hier.get("section")
            })
        
        return {
            "code_name": self.code_name,
            "total_articles": len(articles),
            "articles": articles
        }

print("ArticleExtractor class loaded!")

In [ ]:
# =============================================================================
# ARABERT CLASSIFIER (Inference Provider)
# =============================================================================

class AraBERTClassifier:
    """AraBERT classifier for legal_domain and legal_type."""
    
    def __init__(self):
        self.classifier = None
        self.available = False
        
        if TRANSFORMERS_AVAILABLE:
            try:
                print(f"Loading AraBERT model: {ARABERT_MODEL}")
                self.classifier = pipeline(
                    "text-classification",
                    model=ARABERT_MODEL,
                    tokenizer=ARABERT_MODEL
                )
                self.available = True
                print("✓ AraBERT loaded successfully")
            except Exception as e:
                print(f"AraBERT init failed: {e}")
    
    def predict(self, text: str, task: str) -> dict:
        """Predict legal_domain or legal_type."""
        if not self.available:
            return {"prediction": None, "confidence": 0.0}
        
        try:
            truncated = text[:512]
            results = self.classifier(truncated)
            
            if results and len(results) > 0:
                return {
                    "prediction": results[0].get("label"),
                    "confidence": results[0].get("score", 0.0)
                }
        except Exception as e:
            print(f"AraBERT prediction error: {e}")
        
        return {"prediction": None, "confidence": 0.0}

# Initialize AraBERT
arabert = AraBERTClassifier()

In [ ]:
# =============================================================================
# OPENROUTER PROVIDER (LLM Inference)
# =============================================================================

class OpenRouterProvider:
    """OpenRouter API provider for LLM tasks."""
    
    def __init__(self, api_key: str = None):
        self.model = OPENROUTER_MODEL
        self.available = False
        
        self.token = api_key or OPENROUTER_API_KEY
        if not self.token or self.token == "sk-or-v1-...":
            print("⚠️ OpenRouter API key not set!")
            return
        
        self.api_url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.token}",
            "Content-Type": "application/json"
        }
        self.available = True
        print(f"✓ OpenRouter initialized with model: {self.model}")
    
    def extract(self, article_text: str) -> dict:
        """Extract structured data using LLM."""
        if not self.available:
            return {"error": "OpenRouter not available"}
        
        prompt = f"""You are a legal information extraction system specialized in Tunisian law.

TASK: Extract structured metadata from the Arabic legal article below.

FIELDS TO EXTRACT:
- legal_action: Type of legal action (e.g., "تجريم", "عقوبة", "إجراء", null if not applicable)
- crime_type: Type of crime if specified (e.g., "سرقة", "احتيال", null if not a crime article)
- banking_related: Is this related to banking/financial sector? (true/false)
- key_concepts: Array of key legal concepts (e.g., ["الدعوى العمومية", "العقوبة"])
- sanctions: Array of sanctions (e.g., ["سجن", "غرامة"], empty if none)
- procedures: Array of legal procedures mentioned
- rights: Array of rights mentioned
- obligations: Array of obligations mentioned

RULES:
- Do NOT invent information.
- Extract only what is explicitly stated.
- Keep legal terminology in Arabic.
- Output ONLY valid JSON (no markdown, no explanation).

OUTPUT FORMAT:
{
  "legal_action": null,
  "crime_type": null,
  "banking_related": false,
  "key_concepts": [],
  "sanctions": [],
  "procedures": [],
  "rights": [],
  "obligations": []
}

ARTICLE TEXT:
{article_text}

OUTPUT JSON ONLY:"""
        
        try:
            payload = {
                "model": self.model,
                "messages": [
                    {"role": "user", "content": prompt[:3000]}
                ],
                "temperature": 0.1,
                "max_tokens": 800
            }
            
            import requests
            response = requests.post(
                self.api_url,
                headers=self.headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                if "choices" in result and len(result["choices"]) > 0:
                    generated = result["choices"][0]["message"]["content"]
                    return repair_json(generated)
            else:
                print(f"API Error: {response.status_code}")
                return {"error": f"HTTP {response.status_code}"}
        
        except Exception as e:
            print(f"OpenRouter error: {e}")
            return {"error": str(e)}

# Initialize OpenRouter
openrouter = OpenRouterProvider()

In [ ]:
# =============================================================================
# MAIN PIPELINE - Dual Input Support (MD or JSON)
# =============================================================================

def run_pipeline(input_source, code_name: str = None, use_llm: bool = True):
    """
    Run full pipeline: Input (MD or JSON) → Extract → LLM Enhance → Rich JSON
    
    Args:
        input_source: Path to MD file, JSON file, or JSON string
        code_name: Name of legal code (optional)
        use_llm: Whether to use OpenRouter for enhancement
    """
    
    results = {
        "code_name": code_name or "unknown",
        "total_articles": 0,
        "articles": [],
        "metadata": {
            "pipeline_version": "v7.1",
            "processing_date": time.strftime("%Y-%m-%d %H:%M:%S"),
            "llm_provider": "OpenRouter" if use_llm and openrouter.available else None
        }
    }
    
    # === STEP 1: INPUT HANDLING (MD or JSON) ===
    print("="*60)
    print("STEP 1: Loading input...")
    print("="*60)
    
    input_path = Path(input_source)
    
    if input_path.suffix == ".md":
        # Process MD file
        print(f"Processing MD file: {input_path.name}")
        extractor = ArticleExtractor(str(input_path))
        data = extractor.extract_all()
        articles = data["articles"]
        results["code_name"] = data["code_name"]
        print(f"✓ Extracted {len(articles)} articles from MD")
        
    elif input_path.suffix == ".json":
        # Load existing JSON
        print(f"Loading JSON file: {input_path.name}")
        with open(input_path, encoding="utf-8") as f:
            data = json.load(f)
        
        # Handle both single file and batch format
        if "articles" in data:
            articles = data["articles"]
            results["code_name"] = data.get("code_name", code_name or input_path.stem)
        else:
            # Assume it's already an articles list
            articles = data if isinstance(data, list) else []
            results["code_name"] = code_name or input_path.stem
        
        print(f"✓ Loaded {len(articles)} articles from JSON")
    
    else:
        print(f"❌ Unsupported input format: {input_path.suffix}")
        return results
    
    results["total_articles"] = len(articles)
    
    # === STEP 2: LLM ENHANCEMENT ===
    if use_llm and openrouter.available:
        print("\n" + "="*60)
        print("STEP 2: LLM Enhancement (OpenRouter)")
        print("="*60)
        
        for idx, article in enumerate(articles):
            print(f"\nProcessing article {idx+1}/{len(articles)} (Article {article.get('article_number', '?')}")
            
            # Call OpenRouter
            extracted = openrouter.extract(article.get("content", ""))
            
            # Merge extracted fields
            if "error" not in extracted:
                article["llm_metadata"] = extracted
            else:
                article["llm_metadata"] = {
                    "legal_action": None,
                    "crime_type": None,
                    "banking_related": False,
                    "key_concepts": [],
                    "sanctions": [],
                    "procedures": [],
                    "rights": [],
                    "obligations": []
                }
            
            # Rate limiting
            if (idx + 1) % BATCH_SIZE == 0:
                print(f"  Pausing {DELAY_BETWEEN_REQUESTS}s...")
                time.sleep(DELAY_BETWEEN_REQUESTS)
        
        print(f"\n✓ LLM enhancement complete for {len(articles)} articles")
    
    else:
        print("\n⚠️ Skipping LLM enhancement (not available or disabled)")
        for article in articles:
            article["llm_metadata"] = {
                "legal_action": None,
                "crime_type": None,
                "banking_related": False,
                "key_concepts": [],
                "sanctions": [],
                "procedures": [],
                "rights": [],
                "obligations": []
            }
    
    # === STEP 3: BUILD RICH JSON (articles.json format) ===
    print("\n" + "="*60)
    print("STEP 3: Building Rich JSON...")
    print("="*60)
    
    rich_articles = []
    for idx, article in enumerate(articles):
        # Build ID
        article_id = f"{results['code_name']}__{article.get('article_number', idx)}"
        
        # Extract LLM metadata
        llm_meta = article.get("llm_metadata", {})
        
        # Build legal_features
        legal_features = {
            "has_sanction": len(llm_meta.get("sanctions", [])) > 0,
            "has_deadline": False,
            "topics": llm_meta.get("key_concepts", [])[:5],
            "keywords": llm_meta.get("key_concepts", [])
        }
        
        # Build relations (simplified for Colab)
        relations = {
            "refers_to": [],
            "modified_by": [],
            "similar_articles": []
        }
        
        # Build rich article
        rich_article = {
            "id": article_id,
            "article_number": article.get("article_number", "?"),
            "content": article.get("content", ""),
            "code": {
                "name": results["code_name"],
                "domain": llm_meta.get("legal_action", "unknown"),
                "type": "code"
            },
            "hierarchy": {
                "book": article.get("book"),
                "title": article.get("title"),
                "section": article.get("section")
            },
            "metadata": {
                "language": "ar",
                "is_active": True
            },
            "legal_features": legal_features,
            "relations": relations,
            "semantic": {
                "topic": llm_meta.get("legal_action"),
                "importance": 1.0
            }
        }
        
        rich_articles.append(rich_article)
    
    results["articles"] = rich_articles
    print(f"✓ Built {len(rich_articles)} rich articles")
    
    # === STEP 4: CREATE CHUNKS ===
    print("\n" + "="*60)
    print("STEP 4: Creating Chunks...")
    print("="*60)
    
    chunks = []
    for article in rich_articles:
        # Simple chunking: split by sentences
        content = article["content"]
        sentences = re.split(r'[.!?]\s+', content)
        
        chunk_size = 500  # characters per chunk
        current_chunk = ""
        chunk_idx = 0
        
        for sentence in sentences:
            if len(current_chunk) + len(sentence) < chunk_size:
                current_chunk += sentence + ". "
            else:
                if current_chunk:
                    chunks.append({
                        "chunk_id": f"{article['id']}__chunk_{chunk_idx}",
                        "article_id": article["id"],
                        "text": current_chunk.strip(),
                        "semantic": article.get("semantic", {})
                    })
                    chunk_idx += 1
                current_chunk = sentence + ". "
        
        # Last chunk
        if current_chunk:
            chunks.append({
                "chunk_id": f"{article['id']}__chunk_{chunk_idx}",
                "article_id": article["id"],
                "text": current_chunk.strip(),
                "semantic": article.get("semantic", {})
            })
    
    results["chunks"] = chunks
    print(f"✓ Created {len(chunks)} chunks")
    
    print("\n" + "="*60)
    print("PIPELINE COMPLETE!")
    print("="*60)
    print(f"Total articles: {len(rich_articles)}")
    print(f"Total chunks: {len(chunks)}")
    print(f"Code: {results['code_name']}")
    
    return results

print("Pipeline function loaded!")

In [ ]:
# =============================================================================
# RUN PIPELINE - User Input Required
# =============================================================================

# === UPLOAD FILE TO COLAB ===
from google.colab import files

print("Upload your input file (MD or JSON):")
uploaded = files.upload()

# Get uploaded filename
if uploaded:
    input_file = list(uploaded.keys())[0]
    print(f"\nUploaded: {input_file}")
else:
    print("No file uploaded!")
    input_file = None

# === RUN PIPELINE ===
if input_file:
    print("\n" + "="*60)
    print("STARTING PIPELINE...")
    print("="*60)
    
    # Uncomment to skip LLM (for testing without API key)
    # results = run_pipeline(input_file, use_llm=False)
    
    results = run_pipeline(input_file, use_llm=True)
    
    print("\n✓ Pipeline complete!")
else:
    print("Please upload a file first.")

In [ ]:
# =============================================================================
# DISPLAY & DOWNLOAD RESULTS
# =============================================================================

if 'results' in locals():
    # Display summary
    print("="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    print(f"Code Name: {results['code_name']}")
    print(f"Total Articles: {results['total_articles']}")
    print(f"Total Chunks: {len(results.get('chunks', []))}")
    
    # Show sample article
    if results.get("articles"):
        sample = results["articles"][0]
        print("\n" + "-"*60)
        print("SAMPLE ARTICLE (first one):")
        print("-"*60)
        print(f"ID: {sample['id']}")
        print(f"Article Number: {sample['article_number']}")
        print(f"Code: {sample['code']['name']}")
        print(f"Content (first 200 chars): {sample['content'][:200]}...")
        print(f"Legal Features: {json.dumps(sample['legal_features'], ensure_ascii=False)}")
        print(f"Relations: {json.dumps(sample['relations'], ensure_ascii=False)}")
    
    # Save to JSON files
    output_articles = "/content/articles.json"
    output_chunks = "/content/chunks.json"
    
    with open(output_articles, "w", encoding="utf-8") as f:
        json.dump(results["articles"], f, ensure_ascii=False, indent=2)
    
    with open(output_chunks, "w", encoding="utf-8") as f:
        json.dump(results.get("chunks", []), f, ensure_ascii=False, indent=2)
    
    print("\n" + "="*60)
    print("FILES SAVED!")
    print("="*60)
    print(f"articles.json: {output_articles}")
    print(f"chunks.json: {output_chunks}")
    
    # Download files
    print("\nDownloading files...")
    files.download(output_articles)
    files.download(output_chunks)
    
    print("\n✓ Done! Check your Downloads folder.")
else:
    print("No results to display. Run the pipeline first.")

In [ ]:
# =============================================================================
# VERIFICATION - Check output format matches v7.1 schema
# =============================================================================

def verify_output(articles, chunks):
    """Verify output matches expected schema."""
    print("="*60)
    print("OUTPUT VERIFICATION")
    print("="*60)
    
    # Check articles schema
    required_article_fields = ["id", "article_number", "content", "code", "legal_features", "relations"]
    required_code_fields = ["name", "domain", "type"]
    required_features_fields = ["has_sanction", "has_deadline", "topics", "keywords"]
    
    print("\nChecking articles...")
    issues = 0
    for idx, art in enumerate(articles[:5]):  # Check first 5
        for field in required_article_fields:
            if field not in art:
                print(f"  ❌ Article {idx}: Missing field '{field}'")
                issues += 1
        
        if "code" in art:
            for field in required_code_fields:
                if field not in art["code"]:
                    print(f"  ❌ Article {idx}: Missing code.{field}")
                    issues += 1
        
        if "legal_features" in art:
            for field in required_features_fields:
                if field not in art["legal_features"]:
                    print(f"  ❌ Article {idx}: Missing legal_features.{field}")
                    issues += 1
    
    # Check chunks schema
    required_chunk_fields = ["chunk_id", "article_id", "text", "semantic"]
    
    print("\nChecking chunks...")
    for idx, chunk in enumerate(chunks[:5]):  # Check first 5
        for field in required_chunk_fields:
            if field not in chunk:
                print(f"  ❌ Chunk {idx}: Missing field '{field}'")
                issues += 1
    
    print("\n" + "="*60)
    if issues == 0:
        print("✓ All checks passed! Output matches v7.1 schema.")
    else:
        print(f"⚠️ Found {issues} issues. Check output format.")
    print("="*60)

# Run verification
if 'results' in locals():
    verify_output(results.get("articles", []), results.get("chunks", []))
    
    # Display statistics
    print("\n" + "="*60)
    print("STATISTICS")
    print("="*60)
    
    articles = results.get("articles", [])
    if articles:
        # Count articles with sanctions
        with_sanctions = sum(1 for a in articles if a.get("legal_features", {}).get("has_sanction"))
        print(f"Articles with sanctions: {with_sanctions}/{len(articles)}")
        
        # Count by legal action
        actions = {}
        for a in articles:
            action = a.get("legal_features", {}).get("topics", [None])[0]
            if action:
                actions[action] = actions.get(action, 0) + 1
        
        print("\nTop legal topics:")
        for topic, count in sorted(actions.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"  - {topic}: {count}")
else:
    print("No results to verify. Run pipeline first.")